In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("tugas7-Pertemuan7") \
    .master("local[*]") \
    .getOrCreate()

# Mengatur log agar hanya menampilkan pesan ERROR
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 4.2.0


In [5]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/imam/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/imam/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/imam/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/imam/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [9]:
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/imam/tugas7/raw/tugas7_transaksi.csv",
    header=True, inferSchema=True
)
df_cabang = spark.read.json("hdfs://localhost:9000/user/imam/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/imam/tugas7/raw/tugas7_menu.csv",
    header=True, inferSchema=True
)

print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Transaksi: 4000 | Cabang: 3 | Menu: 15


In [27]:
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                           .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [29]:
# Meringkas total penjualan per kombinasi nama_cabang + menu terlebih dahulu
ringkasan_nama_cabang_menu = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Menerapkan window function untuk mengambil top-2 produk pada setiap cabang
window_nama_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())
top_menu = ringkasan_nama_cabang_menu.withColumn("peringkat", row_number().over(window_nama_cabang)) \
                                    .filter(col("peringkat") <= 2)

top_menu.orderBy("nama_cabang", "peringkat").show(10)

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



In [21]:
df_gabungan.createOrReplaceTempView("gabungan")

query = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')
query.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [31]:
!hdfs dfs -mkdir -p /user/imam/tugas7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/imam/tugas7/processed/laporan"
)

print("Laporan berhasil disimpan ke HDFS dalam format Parquet. Shuuuu")
!hdfs dfs -ls /user/imam/tugas7/processed/laporan

Laporan berhasil disimpan ke HDFS dalam format Parquet. Shuuuu


Found 4 items
-rw-r--r--   3 imam supergroup          0 2026-09-30 22:52 /user/imam/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - imam supergroup          0 2026-09-30 22:52 /user/imam/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - imam supergroup          0 2026-09-30 22:52 /user/imam/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - imam supergroup          0 2026-09-30 22:52 /user/imam/tugas7/processed/laporan/kategori_menu=Non-Kopi


In [34]:
# Tabel Ringkasan metrik per cabang menggunakan Spark SQL
df_final = spark.read.parquet("hdfs://localhost:9000/user/imam/tugas7/processed/laporan")

ringkasan_eksekutif = df_final.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif.show()

+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+



**Analisis Kinerja Terbaik**
Berdasarkan rekapitulasi data di atas, cabang yang berkinerja terbaik adalah **Kopi Nusantara Semarang** dengan perolehan total penjualan tertinggi mencapai Rp90.542.000 dari 1.393 jumlah transaksi.

**Rekomendasi Strategi Promosi:**
Untuk meningkatkan performa di cabang yang dengan total penjualan terlemah (Kopi Nusantara Yogyakarta), manajemen disarankan untuk lebih gencar mempromosikan produk terlaris **makanan** melalui paket **bundling (pakethemat bersama kopi)**. Alasannya, peningkatan transaksi melalui produk makanan bonus minuman kopi dapat menarik segmen konsumen yang lebih luas untuk lebih menghemat uang, sehingga mampu menaikkan total pendapatan keseluruhan agar dapat bersaing dengan cabang Semarang pada waktu berikutnya.